# 📈 大模型评测：从“玄学”到“科学”

在大模型（LLM）的研发过程中，**“如何评价一个模型的好坏”** 是一个比训练模型更具挑战性的课题。如果说模型训练是“考试前刷题”，那么评测就是“那张决定命运的试卷”。

根据 **HBPU_AI_C10** 课程大纲，我们将在这个 notebook 中完整体验以下评测方法：
1. **客观准确率** (Zero-shot / Few-shot)
2. **粒度指标** (Exact Match / F1 Score)
3. **困惑度** (Perplexity)
4. **自动主观评分** (LLM-as-a-judge)
5. **多维雷达报告** (Radar Analysis)

---

## 🛠️ 1. 环境初始化：高可靠加载 Qwen2

根据我们的工程准则，我们将使用“自愈式加载器”。它会自动检查本地 `models/` 目录下的模型完整性。如果文件缺失或损坏，它会尝试通过镜像站补全。

我们将使用本地的 **Qwen2-0.5B-Instruct** 作为评测对象。

In [ ]:
import os
import torch
import torch.nn.functional as F
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from openai import OpenAI

# 设置镜像站加速 (国内环境必备)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def load_qwen_robustly(repo_id="Qwen/Qwen2-0.5B-Instruct"):
    """带有物理校验的模型加载器"""
    model_name = repo_id.split("/")[-1]
    local_path = Path("models") / model_name
    
    print(f"🔍 正在校验物理层：{local_path}...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_path),
        local_dir_use_symlinks=False,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
        max_workers=4
    )
    
    tokenizer = AutoTokenizer.from_pretrained(str(local_path), use_fast=False)
    model = AutoModelForCausalLM.from_pretrained(
        str(local_path),
        device_map="cpu",      # 强制 CPU 运行以兼容教学环境
        torch_dtype="auto"
    )
    return model, tokenizer

model, tokenizer = load_qwen_robustly()
print("✅ Qwen2-0.5B 已成功载入物理内存。")

## 📚 2. 大模型评测的“工具箱”

### 1️⃣ 客观评测 (Objective)
- **Accuracy / EM (Exact Match)**: 统计对错。
- **F1 Score**: 衡量预测与参考答案的文本重叠度（Token/Char 级别）。

### 2️⃣ 语言建模评测 (Modeling)
- **PPL (Perplexity, 困惑度)**。用于衡量模型预测文本的确定性. PPL 越低，说明模型觉得这段文字越“通顺”。

### 3️⃣ 主观评测 (Subjective)
- **LLM-as-a-judge**。利用一个更强大的模型（判官）来给本地模型（考生）的生成结果打分。

## 📊 3. 准备评测数据集：Mini-C-Eval

我们先定义 5 道经典的中文知识题。

In [ ]:
eval_data = [
    {"question": "《史记》的作者是谁？", "choices": {"A": "司马光", "B": "司马迁", "C": "班固", "D": "鲁迅"}, "answer": "B"},
    {"question": "光在真空中传播的速度大约是多少？", "choices": {"A": "30万公里/秒", "B": "30万公里/小时", "C": "100万公里/秒", "D": "340米/秒"}, "answer": "A"},
    {"question": "下列哪个不是中国的四大发明？", "choices": {"A": "火药", "B": "造纸术", "C": "蒸汽机", "D": "指南针"}, "answer": "C"},
    {"question": "水的分子式是？", "choices": {"A": "CO2", "B": "H2O", "C": "NaCl", "D": "O2"}, "answer": "B"},
    {"question": "人身上共有多少块骨头？", "choices": {"A": "106", "B": "256", "C": "206", "D": "186"}, "answer": "C"}
]

def format_question(item):
    prompt = f"问题：{item['question']}\n"
    for key, val in item['choices'].items():
        prompt += f"{key}. {val}\n"
    prompt += "答案："
    return prompt

## 🎯 4. 传统指标：从精确到语义

在评估模型（尤其是问答、选择题等任务）时，简单的准确率往往不够。我们需要以下核心指标来衡量模型输出的质量。

### 1️⃣ Exact Match (EM, 全量匹配)
**定义**：预测答案必须与参考答案**完全一致**。
- **计算逻辑**：如果 `预测值 == 参考答案`，则得分 1，否则得分 0。
- **教学案例**：
    - **参考答案**：`"司马迁"` 
    - **模型预测 A**：`"司马迁"` -> **EM = 1 ✅**
    - **模型预测 B**：`"法学家司马迁"` -> **EM = 0 ❌**

---

### 2️⃣ 查准率、查全率与 F1 Score
我们常用 **字符级别 (Character-level) 的 F1 Score** 来衡量文本相似度。

- **查准率 (P)**：说出的字里，有多少是真的？(`重叠/预测总数`)
- **查全率 (R)**：参考答案里，找回来了多少？(`重叠/参考总数`)
- **F1 Score**：两者的平衡点。(`2*P*R / (P+R)`)

**教学案例**：
- **参考答案**：`"北京大学"` (4个字)
- **模型预测**：`"北京"` (2个字)
- **计算**：
    - **Precision** = 2/2 = 1.0 (全对)
    - **Recall** = 2/4 = 0.5 (漏了一半)
    - **F1 Score** ≈ **0.67**

In [ ]:
import transformers
import warnings
from tqdm.auto import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 屏蔽警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning)

def compute_f1_char_level(pred, ref):
    """计算两个字符串的字符级 F1 Score"""
    pred_chars, ref_chars = list(pred), list(ref)
    common_chars = []
    temp_ref = ref_chars.copy()
    for c in pred_chars:
        if c in temp_ref:
            common_chars.append(c)
            temp_ref.remove(c)
    common_len = len(common_chars)
    if common_len == 0: return 0.0
    precision = common_len / len(pred_chars)
    recall = common_len / len(ref_chars)
    return (2 * precision * recall) / (precision + recall)

def compute_bleu(pred, ref):
    """计算简易 BLEU 分数 (保留定义供后续实验使用)"""
    reference = [list(ref)]
    candidate = list(pred)
    smoothie = SmoothingFunction().method1
    return sentence_bleu(reference, candidate, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)

def get_model_prediction(full_prompt, model, tokenizer):
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cpu")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=5, 
            do_sample=False, 
            temperature=None,
            top_p=None, 
            top_k=None,
            pad_token_id=tokenizer.eos_token_id
        )
    raw_res = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return raw_res[0].upper() if len(raw_res) > 0 else "N/A"

print("🚀 正在运行综合能力测试 (Mini-C-Eval)...\n")

em_list, f1_list = [], []
results_table = []

# 评测循环
for i, item in enumerate(tqdm(eval_data, desc="正在测评", leave=False)):
    pred = get_model_prediction(format_question(item), model, tokenizer)
    correct = item['answer']
    
    is_correct = (pred == correct)
    em_list.append(1.0 if is_correct else 0.0)
    f1_list.append(compute_f1_char_level(pred, correct))
    
    status = "✅" if is_correct else "❌"
    results_table.append(f"#{i+1:<4} {pred:<14} {correct:<14} {status}")

# 统一打印结果
print(f"{'ID':<5} {'模型预测':<12} {'参考答案':<12} {'判定':<5}")
print("-" * 40)
for row in results_table:
    print(row)

# 统计最终得分 (此处不展示 BLEU)
print(f"\n🏆 [综合评分汇总]:")
print(f"- Exact Match (EM): {sum(em_list)/len(em_list):.1%}")
print(f"- Avg F1 Score: {sum(f1_list)/len(f1_list):.2f}")

### 3️⃣ BLEU Score (机器翻译的基准)
**定义**：通过 N-gram 匹配来评价预测文本与参考文本的相似度，尤其关注**句子的连贯性**。

#### 💡 核心逻辑：为什么 F1 不够，需要 BLEU？
BLEU 引入了**短语匹配 (N-gram)** 和 **短句惩罚 (BP)**。

**案例 A：连贯性测试 (N-gram 的作用)**
- **参考内容**：`"今天天气很好"` (1-gram: [今,天,天,气,很,好] | 2-gram: [今天,天天,天气,气很,很好])
- **模型回答**：`"很好今天天气"` 
- **判定分析**：
    - **F1 分数**：**1.0** (字都在，F1 认为它很完美)。
    - **BLEU 评分**：**极低**。因为虽然字对，但 2-gram 里的 `很好今天` 在参考答案里完全找不到。BLEU 识破了它的不连贯。

**案例 B：偷懒测试 (Brevity Penalty 的作用)**
- **参考内容**：`"今天天气很好"` (6 个字符)
- **模型回答**：`"今天"` (2 个字符)
- **判定分析**：
    - **BLEU 评分**：**受罚扣分**。虽然 `今天` 是对的，但由于长度远短于参考答案，BLEU 会自动触发 **BP 惩罚项**，降低最终得分，防止模型通过短回答“刷分”。

- **1-gram**: 衡量词汇准确性。
- **2-gram ~ 4-gram**: 衡量流利度。

### 🧪 指标灵敏度深度实验：F1 vs BLEU
为了让大家直观感受到不同指标对“语序”和“长度”的敏感程度，我们设计了以下对比实验。
我们固定一个 **标准参考答案 (Reference)**，然后构造三个具有代表性的 **模型预测 (Prediction)**：
1. **方案 A (乱序版)**：字全对，但顺序混乱。
2. **方案 B (关键词版)**：只说出核心词。
3. **方案 C (极简版)**：不仅字少，且触发短句惩罚。

In [ ]:
# 指标灵敏度深度剖析 (Case-by-Case Analysis)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference_text = "今天天气很好，适合出去郊游。"

test_cases = [
    {"name": "方案 A (乱序版)", "pred": "适合出去很好今天天气郊游。"},
    {"name": "方案 B (关键词版)", "pred": "今天天气还可以，去散步吧。"},
    {"name": "方案 C (极简/偷懒版)", "pred": "天气很好。"}
]

print(f"📌 标准参考 (Reference): {reference_text}")
print("=" * 60)

for case in test_cases:
    f1 = compute_f1_char_level(case["pred"], reference_text)
    bleu = compute_bleu(case["pred"], reference_text)
    
    # 根据案例类型定制深度观察
    if "乱序" in case["name"]:
        observation = "可以看到，虽然字符匹配度极高(F1=0.96)，但由于语义完全不通顺，BLEU(0.80)相比理论满分有了显著回落，成功识破了“语序乱套”。"
    elif "关键词" in case["name"]:
        observation = "这是最常见的场景。模型捕捉到了部分核心意图，F1 和 BLEU 均处于中等水平，客观反映了语义重叠度。"
    else: # 偷懒版
        observation = "尽管‘天气很好’这四个字完全正确，但由于长度严重不足，BLEU 触发了严厉的【短句惩罚 (BP)】。此时 BLEU(0.14) 的反馈比 F1(0.53) 更能体现出回答完整性的缺失。"

    # 结构化打印
    print(f"\n[ {case['name']} ]")
    print(f" └─ 预测内容: {case['pred']}")
    print(f" └─ 指标数据: F1 Score = {f1:.2f} | BLEU Score = {bleu:.2f}")
    print(f" └─ 深度解析: {observation}")
    print("-" * 60)


## 🧪 5. 困惑度 (Perplexity, PPL) 评测

PPL 是文本平均负对数似然概率的指数。简单理解：模型在预测下一个词时越“吃惊”，PPL 越高；模型觉得越丝滑，PPL 越低。

**教学重点**：
- **整句 PPL**：代表模型对整条序列的整体接受度。
- **逐字 PPL**：通过条形图，我们可以观察到模型在哪个字词处发生了“逻辑断裂”（PPL 突增），从而直观理解模型的预测行为。

In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.font_manager
import numpy as np

def calculate_token_ppl(text, model, tokenizer):
    """计算序列中每个 Token 的困惑度序列"""
    inputs = tokenizer(text, return_tensors="pt").to("cpu")
    input_ids = inputs["input_ids"]
    labels = input_ids.clone()
    
    with torch.no_grad():
        # 获取模型输出
        outputs = model(input_ids)
        logits = outputs.logits
        
        # Shift 逻辑：模型在位置 i 预测的是位置 i+1 的 Token
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        # 计算逐个位置的交叉熵损失
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        # 转换为 PPL (e^loss)
        token_ppls = torch.exp(token_losses).tolist()
        
        # 字符解码
        tokens = [tokenizer.decode(t) for t in shift_labels[0]]
        
    return tokens, token_ppls

def plot_ppl_comparison(text_fluent, text_broken, model, tokenizer):
    """绘图：对比通顺与乱序文本的逐字困惑度"""
    # 设置中文显示环境
    target_fonts = ['SimHei', 'Arial Unicode MS', 'Microsoft YaHei', 'DejaVu Sans']
    available_fonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
    for font in target_fonts:
        if font in available_fonts:
            plt.rcParams['font.sans-serif'] = [font]
            break
    plt.rcParams['axes.unicode_minus'] = False
    
    tokens_f, ppls_f = calculate_token_ppl(text_fluent, model, tokenizer)
    tokens_b, ppls_b = calculate_token_ppl(text_broken, model, tokenizer)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # 场景 1：通顺文本
    ax1.bar(range(len(tokens_f)), ppls_f, color='#2ecc71', alpha=0.7, edgecolor='black')
    ax1.set_xticks(range(len(tokens_f)))
    ax1.set_xticklabels(tokens_f)
    ax1.set_title(f"通顺文本逐字 PPL (均值: {np.mean(ppls_f):.2f})", fontsize=12)
    ax1.set_ylabel("Perplexity")
    ax1.grid(axis='y', alpha=0.3)
    
    # 场景 2：乱序文本
    ax2.bar(range(len(tokens_b)), ppls_b, color='#e74c3c', alpha=0.7, edgecolor='black')
    ax2.set_xticks(range(len(tokens_b)))
    ax2.set_xticklabels(tokens_b)
    ax2.set_title(f"乱序文本逐字 PPL (均值: {np.mean(ppls_b):.2f})", fontsize=12)
    ax2.set_ylabel("Perplexity")
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 运行比较测试
plot_ppl_comparison(
    "今天天气真好，我们一起去公园散步吧。", 
    "今天天气真好，去散步一起公园我们好。", 
    model, tokenizer
)

## ⚖️ 6. LLM-as-a-judge：语义化的“金牌裁判”

在处理非结构化任务（如创意写作、逻辑分析）时，传统的字符匹配（EM）彻底失效。我们需要借助更强大的大模型（如 Qwen-Plus 或 DeepSeek）作为“裁判”。

---

### 💡 为了让LLM当好裁判，我们需要System Prompt。 那么，什么是 System Prompt呢？

在后方的代码实现中，你会发现在调用 API 时有一个特殊的 `role: "system"`。这是大模型开发中极其实用的技术。

#### 设置目的：身份锚定
默认情况下，模型是一个通用的助手。但在评测场景下，我们需要它变成一个**铁面无私的裁判**。
*   **目的**：通过 System Prompt 强制赋予模型一个专业角色，排除干扰。

#### 作用：规则指令集
System Prompt 相当于给模型下达的“最高指令”。在 `call_judge` 中：
*   它规定了**评分维度**（准确性、语言质量）。
*   它强制了**输出格式**（必须是特定的 JSON 结构），这是程序自动解析数据的关键。

#### 调用方式
在 OpenAI 兼容的 API 规范中，对话是由消息列表构成的。System Prompt 永远位于**列表的第一位**：
```python
messages = [
    {"role": "system", "content": "你是一个公正的裁判..."}, # 系统定义：基调与规则
    {"role": "user", "content": "请评价这个回答：..."}      # 用户输入：具体任务
]
```

#### 影响：减少幻觉与增加一致性
如果没有 System Prompt，模型可能会像普通聊天那样说“你好，我觉得这个挺好的...”。
设置了它之后，模型会产生**“指令遵循惯性”**，直接跳过废话，给出我们预期的专业评分。

---

**LLM 作为裁判的核心优势：**
- **语义理解**：识别出“我认为选B”与参考答案“B”在语义上等价。
- **提供理由**：它不仅告诉你得了几分，还会通过理由支撑其判定逻辑。

In [ ]:
# ============================================================
# 👇 把你的 API Key 填在下面的引号里
# ============================================================
API_KEY = ""
API_PROVIDER = "DashScope" # 可选: "DashScope" 或 "SiliconFlow"

if API_PROVIDER == "DashScope":
    BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
    JUDGE_MODEL = "qwen3.5-plus"
else:
    BASE_URL = "https://api.siliconflow.cn/v1"
    JUDGE_MODEL = "deepseek-ai/DeepSeek-V3"

judge_client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def call_judge(question, response):
    system_prompt = "你是一个公正的评测对裁判。请根据【准确性】和【语言质量】给回复打分（1-10分）。\n输出严格JSON: {\"score\": X, \"reason\": \"...\"}"
    user_content = f"问题：{question}\n待评测回答：{response}"
    try:
        res = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_content}],
            response_format={ "type": "json_object" } if API_PROVIDER == "DashScope" else None
        )
        return json.loads(res.choices[0].message.content)
    except Exception as e:
        return {"score": 0, "reason": "API失效"}

---
LLM-as-a-judge 实战案例展示

In [ ]:
judge_test_cases = [
    {
        "type": "语义选择题 (语义优先)",
        "question": "《史记》的作者是谁？ (A. 司马光, B. 司马迁, C. 班固)",
        "response": "我认为这道题应该选 B，因为司马迁倾注一生心血完成了这部史家之绝唱。"
    },
    {
        "type": "创意写作 (开放评价)",
        "question": "请通过描写“风”来表达程序员的孤独，50字以内。",
        "response": "深夜的北风穿透了百叶窗，敲击着机械键盘的清脆声，在空荡的机房里漫无目的地回响。"
    },
    {
        "type": "逻辑推理 (正误识别)",
        "question": "已知 A > B 且 B > C，那么 A 和 C 的关系是？",
        "response": "根据不等式的传递性，我们可以推断出 A 一定小于 C。"
    }
]

print("🔍 正在启动 AI 裁判评价流程...\n")

for i, case in enumerate(judge_test_cases):
    print(f"### 案例 {i+1}: {case['type']} ###")
    print(f"【原题内容】: {case['question']}")
    print(f"【候选回答】: {case['response']}")
    
    # 模拟 API 调用裁判
    result = call_judge(case['question'], case['response'])
    
    print(f"【裁判评分】: ⭐ {result.get('score', 0)} / 10")
    print(f"【裁判理由】: {result.get('reason', 'N/A')}")
    print("-" * 80)
    print()